# OBLITERATUS — One-Click Abliteration

**SOTA refusal removal** running on free Colab GPU. SVD multi-direction extraction, norm-preserving projection, iterative refinement.

Based on: Arditi et al. (2024) | Gabliteration (arXiv:2512.18901) | grimjim norm-preserving biprojection (2025)

---

**How to use:**
1. Make sure GPU runtime is enabled: `Runtime > Change runtime type > T4 GPU`
2. Set your model and method in the config cell below
3. For gated models, complete the Hugging Face access setup in section 2 before running.
4. Run All (`Runtime > Run all` or `Ctrl+F9`)
5. Download the abliterated model from the output

## 1. Install

In [ ]:
!pip install -q git+https://github.com/elder-plinius/OBLITERATUS.git
!pip install -q accelerate bitsandbytes

import torch
print(f"PyTorch {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

## 2. Configure and prepare model access

Edit the cell below to set your target model and abliteration method.

**The default Llama model is gated.** Open its [model page](https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct), accept the license/request access, and wait for approval on your Hugging Face account. A token alone does not grant access.

Create a **read** token for that account (or a fine-grained token permitted to read the selected model). In Colab, open **Secrets** (the key icon), add a secret named `HF_TOKEN`, and enable **Notebook access**. Never paste a token into a code cell or saved output. Outside Colab, an existing `hf auth login` session or an `HF_TOKEN` environment variable also works.

Ungated public models such as `Qwen/Qwen2.5-7B-Instruct` or `openai-community/gpt2` need no login. Step 3 checks access to the selected model's small `config.json` before constructing the pipeline or downloading weights; if access cannot be confirmed, it stops with setup instructions. After fixing access or changing the model, rerun the configuration and step 3. The check runs again even when step 3 is executed directly.


In [ ]:
#@title Abliteration Config { run: "auto" }

#@markdown ### Target Model
#@markdown Pick a model from the dropdown or paste a custom HuggingFace ID.
MODEL = "meta-llama/Llama-3.1-8B-Instruct" #@param ["meta-llama/Llama-3.1-8B-Instruct", "Qwen/Qwen2.5-7B-Instruct", "mistralai/Mistral-7B-Instruct-v0.3", "google/gemma-2-9b-it", "microsoft/Phi-3.5-mini-instruct", "THUDM/glm-4-9b-chat", "NousResearch/Hermes-3-Llama-3.1-8B", "cognitivecomputations/dolphin-2.9.4-llama3.1-8b", "TinyLlama/TinyLlama-1.1B-Chat-v1.0", "openai-community/gpt2"] {allow-input: true}

#@markdown ### Method
METHOD = "advanced" #@param ["basic", "advanced", "aggressive"]

#@markdown ### Advanced Overrides (leave 0 to use method defaults)
N_DIRECTIONS = 0 #@param {type: "integer"}
REGULARIZATION = 0.0 #@param {type: "number"}
REFINEMENT_PASSES = 0 #@param {type: "integer"}

#@markdown ### Output
OUTPUT_DIR = "abliterated" #@param {type: "string"}

print(f"Model: {MODEL}")
print(f"Method: {METHOD}")
print(f"Output: {OUTPUT_DIR}/")

## 3. Run Abliteration Pipeline

This runs all 6 stages: SUMMON → PROBE → DISTILL → EXCISE → VERIFY → REBIRTH

In [ ]:
import os
from huggingface_hub import get_token, hf_hub_download
from obliteratus.credential_sources import resolve_secret


def check_model_access(model_name):
    # Resolve the same configured sources as the loader, then Hub/Colab login.
    # Only the small config is fetched; force a live check even if it is cached.
    try:
        token = (resolve_secret("HF_TOKEN") or get_token() or "").strip() or None
        hf_hub_download(
            repo_id=model_name, filename="config.json",
            token=token or False, force_download=True,
        )
    except Exception:
        # Hub/credential exceptions may contain sensitive request details.
        raise RuntimeError(
            "Model access could not be confirmed. Check the model ID and your "
            "connection. For gated/private models, accept the license/request "
            "access on the model page and wait for approval; use a read token "
            "from that account. In Colab, add HF_TOKEN in Secrets and enable "
            "Notebook access (outside Colab use hf auth login or HF_TOKEN). "
            "Then rerun this cell, or choose an ungated public model. "
            "No model weights have been loaded."
        ) from None
    if token:
        # Runtime only: keep the loader on the credential that passed the check.
        os.environ["HF_TOKEN"] = token


check_model_access(MODEL)

from obliteratus.abliterate import AbliterationPipeline

# Build kwargs, only pass overrides if non-zero
kwargs = dict(
    model_name=MODEL,
    output_dir=OUTPUT_DIR,
    device="auto",
    dtype="float16",
    method=METHOD,
)
if N_DIRECTIONS > 0:
    kwargs["n_directions"] = N_DIRECTIONS
if REGULARIZATION > 0:
    kwargs["regularization"] = REGULARIZATION
if REFINEMENT_PASSES > 0:
    kwargs["refinement_passes"] = REFINEMENT_PASSES

# Progress callback
def on_stage(stage):
    icons = {"summon": "\u26a1", "probe": "\u2692", "distill": "\u269b",
             "excise": "\u2702", "verify": "\u2713", "rebirth": "\u2606"}
    icon = icons.get(stage.stage, "")
    print(f"\n{'='*60}")
    print(f"{icon}  STAGE: {stage.stage.upper()} — {stage.message}")
    print(f"{'='*60}")

def on_log(msg):
    print(f"  {msg}")

kwargs["on_stage"] = on_stage
kwargs["on_log"] = on_log

pipeline = AbliterationPipeline(**kwargs)
result = pipeline.run()

print(f"\n{'='*60}")
print(f"ABLITERATION COMPLETE")
print(f"Output: {result}")
print(f"{'='*60}")

## 4. Download the Abliterated Model

Run the cell below to zip and download, or upload directly to HuggingFace Hub.

In [ ]:
import os
from pathlib import Path

# Find the output directory
out_path = Path(OUTPUT_DIR)
subdirs = [d for d in out_path.iterdir() if d.is_dir()] if out_path.exists() else []
model_dir = subdirs[0] if subdirs else out_path

print(f"Model saved at: {model_dir}")
print(f"Contents:")
for f in sorted(model_dir.rglob("*")):
    if f.is_file():
        size_mb = f.stat().st_size / 1024**2
        print(f"  {f.relative_to(model_dir)}  ({size_mb:.1f} MB)")

In [ ]:
#@title Option A: Download as ZIP
import shutil
from google.colab import files

zip_name = model_dir.name
shutil.make_archive(zip_name, 'zip', model_dir)
print(f"Downloading {zip_name}.zip ...")
files.download(f"{zip_name}.zip")

In [ ]:
#@title Option B: Push to HuggingFace Hub (opt-in)
#@markdown Enable only when ready to upload. Publishing requires a write token with permission for the destination repo; read access to the source model is separate.
UPLOAD_TO_HUB = False #@param {type: "boolean"}
HF_REPO = "your-username/model-name-abliterated" #@param {type: "string"}

if UPLOAD_TO_HUB:
    from huggingface_hub import HfApi, get_token
    from obliteratus.credential_sources import resolve_secret

    if HF_REPO == "your-username/model-name-abliterated":
        raise ValueError("Set HF_REPO to your destination repository before uploading.")
    upload_token = resolve_secret("HF_TOKEN") or get_token()
    if not upload_token:
        raise RuntimeError("Set a write token in HF_TOKEN Secrets before uploading.")
    api = HfApi(token=upload_token)
    api.create_repo(HF_REPO, exist_ok=True)
    api.upload_folder(
        folder_path=str(model_dir),
        repo_id=HF_REPO,
        repo_type="model",
    )
    print(f"\nUploaded to: https://huggingface.co/{HF_REPO}")
else:
    print("Hub upload skipped. Enable UPLOAD_TO_HUB to publish explicitly.")


## 5. Test the Abliterated Model

Quick sanity check — try some prompts that the original model would refuse.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

print(f"Loading abliterated model from {model_dir}...")
tokenizer = AutoTokenizer.from_pretrained(str(model_dir))
model = AutoModelForCausalLM.from_pretrained(
    str(model_dir),
    torch_dtype=torch.float16,
    device_map="auto",
)

def generate(prompt, max_new_tokens=100):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
        )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Test prompts
test_prompts = [
    "The capital of France is",
    "Explain how a neural network works:",
    "Write a poem about the ocean:",
]

for prompt in test_prompts:
    print(f"\n{'='*60}")
    print(f"PROMPT: {prompt}")
    print(f"{'='*60}")
    print(generate(prompt))

In [ ]:
# Safe access check; credentials never appear in output.
from google.colab import userdata
import requests
try:
    check_token = userdata.get('HF_TOKEN').strip()
    print('HF_TOKEN_AVAILABLE:', bool(check_token))
    try:
        check_response = requests.get('https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct/resolve/main/config.json', headers={'Authorization': 'Bearer ' + check_token}, timeout=30)
        print('MODEL_ACCESS_HTTP:', check_response.status_code)
    except Exception as error:
        print('MODEL_CHECK_ERROR:', type(error).__name__)
except Exception as error:
    print('SECRET_CHECK_ERROR:', type(error).__name__)
finally:
    check_token = None
    check_response = None

## 6. Chat API for DeepSeek Harness
Run after `run-pipeline` completes successfully. The listener implements `/v1/models` and `/v1/chat/completions` on loopback port 8000. A dedicated API key is generated in runtime memory. Built with Llama. External connectivity and the desktop provider must be configured after a live API test. Colab sessions are temporary.

In [ ]:
%pip install -q fastapi uvicorn

In [ ]:
"""Run after the OBLITERATUS pipeline completes. Requires fastapi and uvicorn."""
import asyncio
import json
import os
import secrets
import threading
import time
import uuid
from pathlib import Path

import torch
import uvicorn
from fastapi import FastAPI, Header, HTTPException
from fastapi.responses import StreamingResponse
from pydantic import BaseModel, Field

MODEL_ID = "Llama-3.1-8B-Obliteratus"
api_key = os.environ.get("COLAB_API_KEY") or secrets.token_urlsafe(48)
os.environ["COLAB_API_KEY"] = api_key
if not globals().get("pipeline") or not globals().get("result") or not Path(result).is_dir():
    raise RuntimeError("Complete run-pipeline successfully before starting the API.")
api_model = pipeline.handle.model
api_tokenizer = pipeline.handle.tokenizer
if api_model is None or api_tokenizer is None:
    raise RuntimeError("The pipeline has no loaded model. Reload its saved output.")
api_model.eval()
api = FastAPI(docs_url=None, redoc_url=None)
generation_lock = asyncio.Lock()

class ChatRequest(BaseModel):
    model: str = MODEL_ID
    messages: list[dict]
    max_tokens: int = Field(default=256, ge=1, le=2048)
    temperature: float = Field(default=0.7, ge=0, le=2)
    top_p: float = Field(default=0.9, gt=0, le=1)
    stream: bool = False
    tools: list[dict] | None = None

def authenticate(authorization):
    if not authorization or not secrets.compare_digest(authorization, "Bearer " + api_key):
        raise HTTPException(401, "Invalid API key")

@api.get("/v1/models")
def models(authorization: str | None = Header(None)):
    authenticate(authorization)
    return {"object": "list", "data": [{"id": MODEL_ID, "object": "model", "owned_by": "colab"}]}

def generate(request):
    if not request.messages:
        raise ValueError("messages must contain at least one message")
    for message in request.messages:
        if not isinstance(message.get("content", ""), (str, type(None))):
            raise ValueError("This model accepts text messages")
    options = {"tools": request.tools} if request.tools else {}
    prompt = api_tokenizer.apply_chat_template(
        request.messages, tokenize=False, add_generation_prompt=True, **options
    )
    inputs = api_tokenizer(prompt, return_tensors="pt").to(api_model.device)
    context = getattr(api_model.config, "max_position_embeddings", 8192)
    if inputs["input_ids"].shape[1] + request.max_tokens > context:
        raise ValueError("The request exceeds the model context window")
    sampling = {"do_sample": request.temperature > 0}
    if request.temperature > 0:
        sampling.update(temperature=request.temperature, top_p=request.top_p)
    with torch.inference_mode():
        output = api_model.generate(
            **inputs, max_new_tokens=request.max_tokens,
            pad_token_id=api_tokenizer.eos_token_id, **sampling
        )
    tokens = output[0][inputs["input_ids"].shape[1]:]
    text = api_tokenizer.decode(tokens, skip_special_tokens=True)
    return text, inputs["input_ids"].shape[1], len(tokens)

@api.post("/v1/chat/completions")
async def chat(request: ChatRequest, authorization: str | None = Header(None)):
    authenticate(authorization)
    if request.model != MODEL_ID:
        raise HTTPException(404, "Unknown model")
    async with generation_lock:
        try:
            text, prompt_tokens, completion_tokens = await asyncio.to_thread(generate, request)
        except ValueError as error:
            raise HTTPException(400, str(error)) from error
    message = {"role": "assistant", "content": text}
    finish = "length" if completion_tokens >= request.max_tokens else "stop"
    if request.tools:
        try:
            call = json.loads(text.strip().removeprefix("<|python_tag|>"))
            names = {tool["function"]["name"] for tool in request.tools}
            arguments = call.get("parameters", call.get("arguments"))
            if call.get("name") in names and isinstance(arguments, dict):
                message = {"role": "assistant", "content": None, "tool_calls": [{
                    "id": "call_" + uuid.uuid4().hex,
                    "type": "function", "function": {
                        "name": call["name"], "arguments": json.dumps(arguments)
                    }
                }]}
                finish = "tool_calls"
        except (ValueError, TypeError, AttributeError):
            pass
    base = {"id": "chatcmpl-" + uuid.uuid4().hex, "created": int(time.time()), "model": MODEL_ID}
    if request.stream:
        async def events():
            yield "data: " + json.dumps({**base, "object": "chat.completion.chunk", "choices": [
                {"index": 0, "delta": {**message, **({"tool_calls": [{"index": i, **call} for i, call in enumerate(message["tool_calls"])]} if "tool_calls" in message else {})}, "finish_reason": None}
            ]}) + "\n\n"
            yield "data: " + json.dumps({**base, "object": "chat.completion.chunk", "choices": [
                {"index": 0, "delta": {}, "finish_reason": finish}
            ]}) + "\n\n"
            yield "data: [DONE]\n\n"
        return StreamingResponse(events(), media_type="text/event-stream")
    return {**base, "object": "chat.completion", "choices": [
        {"index": 0, "message": message, "finish_reason": finish}
    ], "usage": {"prompt_tokens": prompt_tokens, "completion_tokens": completion_tokens,
                  "total_tokens": prompt_tokens + completion_tokens}}

# A second execution stops the previous API listener before binding its port.
previous = globals().get("colab_api_server")
if previous:
    previous.should_exit = True
    globals()["colab_api_thread"].join(timeout=15)
    if globals()["colab_api_thread"].is_alive():
        raise RuntimeError("The previous API is still shutting down")
colab_api_server = uvicorn.Server(uvicorn.Config(api, host="127.0.0.1", port=8000, log_level="warning"))
colab_api_thread = threading.Thread(target=colab_api_server.run, daemon=True)
colab_api_thread.start()
print("API listener starting on 127.0.0.1:8000; model " + MODEL_ID)